# Notebook 5b: Null Model Simulation (Final)

This notebook implements the **primary gravity-based null model** used in the paper, simulating counterfactual evacuation destinations weighted by destination population.

**Paper reference:** Figure 2c, 2d — Null model construction and comparison of actual vs. simulated destinations.

> ⚠️ **Note:** The pairwise distance computation is computationally intensive. Load the pre-saved CSV instead of re-running that step.

**Inputs:** `evacuees_for_regs.csv`, `tl_2019_08_bg/` shapefile

**Outputs:** `evacuees_newSCI_simulation.csv`, similarity/connectedness distribution plots


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns

In [ ]:
import itertools
from shapely.geometry import Point
from geopy.distance import geodesic

In [ ]:
np.random.seed(42)

In [ ]:
evacuees = pd.read_csv("evacuess_for_regs.csv")
evacuees.head()

In [ ]:
evacuees.columns

In [ ]:
shapefile_path = 'tl_2019_08_bg'
cbg_geo = gpd.read_file(shapefile_path)

cbg_geo['GEOID'] = cbg_geo['GEOID'].str.lstrip('0')

columns_keep = ["GEOID", "geometry"]
colorado_cbgs = cbg_geo[columns_keep]

colorado_cbgs.head()

In [ ]:
colorado_cbgs['centroid'] = colorado_cbgs['geometry'].centroid

In [ ]:
cbgs = colorado_cbgs[['GEOID', 'centroid']]

In [ ]:
colorado_cbgs

# All possible pairs 
this takes time do not run it just read the saves csv - pairwise in the next section

In [ ]:
pairs = list(itertools.product(cbgs['GEOID'], cbgs['GEOID']))

In [ ]:
pairwise = pd.DataFrame(pairs, columns=['origin', 'destination'])

In [ ]:
pairwise = pairwise.merge(cbgs, left_on='origin', right_on='GEOID').rename(columns={'centroid': 'origin_centroid'})
pairwise = pairwise.merge(cbgs, left_on='destination', right_on='GEOID').rename(columns={'centroid': 'destination_centroid'})


In [ ]:
pairwise['distance'] = pairwise.apply(
    lambda row: geodesic(
        (row['origin_centroid'].y, row['origin_centroid'].x),
        (row['destination_centroid'].y, row['destination_centroid'].x)
    ).meters, axis=1
)

In [ ]:
pairwise.head()

In [ ]:
pairwise.to_csv('pairwise')

# Calculating homophily and similarity For all OD combinations

In [ ]:
pairwise = pd.read_csv('pairwise')

In [ ]:
pairwise.head()

In [ ]:
census_org = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")
census_org.columns

In [ ]:
census = census_org[(census_org['median_household_income'])>0]

In [ ]:
census.head()

In [ ]:
plt.hist(census['median_household_income'])

In [ ]:
census['white_population'] = (census['white_population']  /census['total_population'] )*100
census['black_population'] = (census['black_population']  /census['total_population'] )*100
census['asian_population'] = (census['asian_population']  /census['total_population'] )*100
census['all_others'] = ((census['total_population']-(census['white_population']+census['black_population']+ census['asian_population']))
                                                    /census['total_population'])*100
census['education_atleast_one_degree'] = (census['total_population_25_over']  /census['total_population'] )*100

In [ ]:
columns_keep = ['total_population','total_housing_units','white_population', 'black_population', 'asian_population', 'education_atleast_one_degree', 'unemployment_rate','median_household_income', 'geoid', 'all_others' ]
census = census[columns_keep]
census.head()

In [ ]:
census['geoid'] = census['geoid'].astype(str)
cbg_geo['GEOID'] = cbg_geo['GEOID'].astype(str)

In [ ]:
census_r1 = pd.merge(census, cbg_geo, 
                    left_on='geoid', right_on='GEOID', 
                    how='left').rename(columns={'geometry': 'geometry'})

In [ ]:
census_r1 = gpd.GeoDataFrame(census_r1, geometry='geometry')

In [ ]:
census_r1['area_sq_km'] = census_r1['geometry'].area / 10**6
# Calculate population density (people per square kilometer) - or
census_r1['population_density'] = census_r1['total_population'] / census_r1['area_sq_km']

In [ ]:
census_r1.head()

In [ ]:
census_D1 = census_r1.rename(
    columns=lambda x: 'Or_' + x  if x != 'geoid' else x
)
census_D1.head()

In [ ]:
census_D1['geoid'] = census_D1['geoid'].astype(str)
pairwise['origin'] = pairwise['origin'].astype(str)
pairwise_census = pairwise.merge(census_D1, left_on = "origin", right_on = "geoid", how = "inner")

In [ ]:
census_D2 = census_r1.rename(
    columns=lambda x: 'D1_' + x #if x != 'geoid' else x
)
census_D2.head()

In [ ]:
census_D2['D1_geoid'] = census_D2['D1_geoid'].astype(str)
pairwise_census['destination'] = pairwise_census['destination'].astype(str)
pairwise_census_od = pairwise_census.merge(census_D2, left_on = "destination", right_on = "D1_geoid", how = "inner")

In [ ]:
pairwise_census_od.columns

In [ ]:
pairwise_census_od['edu_diff'] = (pairwise_census_od['Or_education_atleast_one_degree']  - pairwise_census_od['D1_education_atleast_one_degree'] )
pairwise_census_od['emp_diff'] = (pairwise_census_od['Or_unemployment_rate']  - pairwise_census_od['D1_unemployment_rate'] )

In [ ]:
pairwise_census_od['income_diff'] = (pairwise_census_od['Or_median_household_income']  - pairwise_census_od['D1_median_household_income'] )
pairwise_census_od['white_diff'] = (pairwise_census_od['Or_white_population']  - pairwise_census_od['D1_white_population'] )

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

In [ ]:
pairwise_census_od.head()

In [ ]:
origin_columns = [
       'Or_population_density', 
       'Or_white_population', 
       #'Or_all_others',
       'Or_black_population',
       #'Or_asian_population', 
       'Or_education_atleast_one_degree',
       'Or_unemployment_rate', 
       'Or_median_household_income'	   
]

destination_columns = [
        'D1_population_density', 
        'D1_white_population',
        #'D1_all_others',
        'D1_black_population', 
        #'D1_asian_population',
       'D1_education_atleast_one_degree', 
       'D1_unemployment_rate',
       'D1_median_household_income'
]

pairwise_census_od = pairwise_census_od.dropna()

# Normalize the data for origin and destination separately
# scaler = MinMaxScaler()
# normalized_origin = scaler.fit_transform(pairwise_census_od[origin_columns])
# normalized_destination = scaler.fit_transform(pairwise_census_od[destination_columns])
scaler = StandardScaler()
normalized_origin = scaler.fit_transform(pairwise_census_od[origin_columns])
normalized_destination = scaler.fit_transform(pairwise_census_od[destination_columns])

# Computing cosine similarity for each row
homophily_scores = []
for i in range(len(pairwise_census_od)):
    origin_vector = normalized_origin[i].reshape(1, -1)
    destination_vector = normalized_destination[i].reshape(1, -1)
    similarity = cosine_similarity(origin_vector, destination_vector)[0][0]
    homophily_scores.append(similarity)

# Append the homophily scores to the original DataFrame
pairwise_census_od['Similarity'] = homophily_scores
pairwise_census_od.head()

In [ ]:
sns.histplot(x='white_diff', data=pairwise_census_od, color='teal', fill=True)

In [ ]:
#pairwise_census_od.to_csv('pairwise_census_od')

In [ ]:
SCI = pd.read_csv("colorado_SCI_ZIP_CBG.csv")

In [ ]:
duplicates = SCI.duplicated(subset=['cbg_user', 'cbg_fr'], keep=False)
if duplicates.sum() > 0:
    print(f"There are {duplicates.sum()} duplicate rows in SCI based on 'cbg_user' and 'cbg_fr'.")

SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()
print(f"Shape of SCI after removing duplicates: {SCI_unique.shape}")
SCI_unique.head()

In [ ]:
SCI_unique['cbg_user'] = SCI_unique['cbg_user'].astype(str)
SCI_unique['cbg_fr'] = SCI_unique['cbg_fr'].astype(str)
pairwise_census_od['origin'] = pairwise_census_od['origin'].astype(str)
pairwise_census_od['destination'] = pairwise_census_od['destination'].astype(str)

In [ ]:
pairwise_census_od_SCI = pd.merge(
    pairwise_census_od, 
    SCI_unique, 
    left_on=['origin', 'destination'],
    right_on=['cbg_user', 'cbg_fr'], 
    how='left' 
    )
pairwise_census_od_SCI.head()

In [ ]:
columns = ['origin', 'destination', 'distance', 'Similarity','scaled_sci', 'income_diff', 'white_diff']
pairwise_subset = pairwise_census_od_SCI[columns]

In [ ]:
#pairwise_census_od_SCI.to_csv('pairwise_subset.csv.gz')

In [ ]:
pairwise_subset.head()

# Evacuees subset 

In [ ]:
# Define the bin edges and labels as you did
bin_edges = np.arange(0, evacuees['distance_OD'].max() + 20000, 20000)
bin_labels = [f'{int(left)}-{int(right)}' for left, right in zip(bin_edges[:-1], bin_edges[1:])]

# Ensure bins are left-inclusive and avoid losing boundary values with right=False
evacuees['distance_range'] = pd.cut(evacuees['distance_OD'], bins=bin_edges, labels=bin_labels, include_lowest=True, right=False)

# Convert distance_range to string, split to get min and max distance values
evacuees['distance_range'] = evacuees['distance_range'].astype(str)
evacuees['distance_min'] = evacuees['distance_range'].apply(lambda x: int(x.split('-')[0]))
evacuees['distance_max'] = evacuees['distance_range'].apply(lambda x: int(x.split('-')[1]))
evacuees.head()

In [ ]:
evacuees_subset = evacuees[['Unnamed: 0', 'pre_crisis_home_cbg', 'crisis_home_cbg', 'D1_total_population', 'distance_OD', 'distance_range', 'distance_min',	'distance_max']]
evacuees_subset.head()

# Merging evacuees with pairwise

In [ ]:
#pairwise_subset = pd.read_csv('pairwise_subset.csv.gz')

In [ ]:
evacuees_subset['pre_crisis_home_cbg'] = evacuees_subset['pre_crisis_home_cbg'].astype(str)
pairwise_subset['origin'] = pairwise_subset['origin'].astype(str)

In [ ]:
evacuees_subset['crisis_home_cbg'] = evacuees_subset['crisis_home_cbg'].astype(str)
pairwise_subset['destination'] = pairwise_subset['destination'].astype(str)


In [ ]:
merged_df = evacuees_subset.merge(pairwise_subset, left_on='pre_crisis_home_cbg', right_on='origin', how = 'left')

merged_df.head()

In [ ]:
# Now, modify the filtering step to ensure the destination bin is not lost
filtered_df = merged_df[(merged_df['distance_OD'] >= merged_df['distance_min']) & (merged_df['distance_OD'] <= merged_df['distance_max'])]

# Drop unnecessary columns after filtering
filtered_df = filtered_df.drop(['distance_min', 'distance_max'], axis=1)

# Check the filtered dataframe
filtered_df.head()

In [ ]:
filtered_df.shape

In [ ]:
filtered_df['Unnamed: 0'].nunique()

In [ ]:
filtered_df.columns

In [ ]:
census = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")
census.columns

In [ ]:
columns_keep = ['geoid','total_population']
census = census[columns_keep]
census.head()

In [ ]:
census['geoid'] = census['geoid'].astype(str)
filtered_df['destination'] = filtered_df['destination'].astype(str)

In [ ]:
filtered_df = filtered_df.merge(census, left_on = "destination", right_on = "geoid", how = "left")

In [ ]:
filtered_df.columns

In [ ]:
#filtered_df.to_csv('for_simulation.csv.gz')

# Probability

## calculating probabilities for all pairs

In [ ]:
filtered_df['probabilities'] = filtered_df.groupby('Unnamed: 0')['total_population'].transform(lambda x: x / x.sum())
filtered_df.head()

In [ ]:
filtered_df.columns

In [ ]:
#filtered_df.to_csv('for_simulated_final.csv.gz')

In [ ]:
#filtered_df = pd.read_csv('for_simulated_final.csv.gz')
filtered_df.head()

In [ ]:
filtered_df.shape

## Subset of actual destinations

In [ ]:
filtered_df['crisis_home_cbg'] = filtered_df['crisis_home_cbg'].astype(str)
filtered_df['destination'] = filtered_df['destination'].astype(str)

In [ ]:
crisis_home_df = filtered_df[filtered_df['crisis_home_cbg'] == filtered_df['destination']]

In [ ]:
census['geoid'] = census['geoid'].astype(str)
crisis_home_df['destination'] = crisis_home_df['destination'].astype(str)

In [ ]:
crisis_home_df = crisis_home_df.merge(census, left_on = "destination", right_on = "geoid", how = "left")

In [ ]:
crisis_home_df.shape

In [ ]:
crisis_home_df.head()

In [ ]:
crisis_home_df['probabilities_log'] = crisis_home_df['probabilities'].apply(np.log1p)

In [ ]:
crisis_home_df.to_csv("actual_location_revised.csv.gz")

## Subset of random dest weighted by pop probabilities

In [ ]:
filtered_df = filtered_df.groupby('Unnamed: 0').filter(lambda x: x['probabilities'].sum() > 0)

In [ ]:
random_destination_pop_df = filtered_df.groupby('Unnamed: 0').apply(lambda x: x.sample(1, weights=x['probabilities'])).reset_index(drop=True)
#random_destination_pop_df['probabilities_log'] = random_destination_pop_df['probabilities'].apply(np.log1p)
random_destination_pop_df.head()

In [ ]:
random_destination_pop_df.shape

In [ ]:
random_destination_pop_df.columns

In [ ]:
random_destination_pop_df.to_csv("simulated_location_revised.csv.gz")

# PLOT FINAL

In [ ]:
actual_destination = pd.read_csv("evacuees_newSCI_simulation.csv").reset_index(drop=True)
actual_destination.head()

In [ ]:
actual_destination.columns

In [ ]:
actual_destination['Familiarity_log'] = actual_destination['scaled_sci'].apply(np.log1p)
actual_destination['Similarity_log'] = actual_destination['homophily'].apply(np.log1p)
actual_destination.head()

In [ ]:
actual_destination.to_csv("actual_destination_rervised.csv")

In [ ]:
simulated_destination = pd.read_csv("simulated_location_revised.csv.gz")
simulated_destination['Familiarity_log'] = simulated_destination['scaled_sci'].apply(np.log1p)
simulated_destination['Similarity_log'] = simulated_destination['Similarity'].apply(np.log1p)
simulated_destination.head()

In [ ]:
# simulated_destination_withoutextreme = simulated_destination[np.abs(simulated_destination['income_diff'])<1000000]
# actual_destination_withoutextreme = actual_destination[np.abs(actual_destination['income_diff'])<1000000]

In [ ]:
# # simulated_destination['log_income_diff'] = np.log1p(simulated_destination['income_diff'].abs())
# # actual_destination['log_income_diff'] = np.log1p(actual_destination['income_diff'].abs())

# # Plot the histograms with log-scaled data
# sns.histplot(x='income_diff', data=simulated_destination_withoutextreme, color='orange', fill=True, alpha=0.5, label='Simulated')
# sns.histplot(x='income_diff', data=actual_destination_withoutextreme, color='teal', fill=True, alpha=0.5, label='Actual')

# plt.legend 
# plt.show()

In [ ]:
# np.mean(simulated_destination_withoutextreme['income_diff']),np.std(simulated_destination_withoutextreme['income_diff'])/np.sqrt(len(simulated_destination_withoutextreme))

In [ ]:
# np.mean(actual_destination_withoutextreme['income_diff']),np.std(actual_destination_withoutextreme['income_diff'])/np.sqrt(len(actual_destination_withoutextreme))

In [ ]:
# simulated_destination['log_income_diff'] = np.where(simulated_destination['income_diff'] >= 0,
#                                                     np.log1p(simulated_destination['income_diff']),
#                                                     -np.log1p(-simulated_destination['income_diff']))

# actual_destination['log_income_diff'] = np.where(actual_destination['income_diff'] >= 0,
#                                                  np.log1p(actual_destination['income_diff']),
#                                                  -np.log1p(-actual_destination['income_diff']))
# sns.histplot(x='log_income_diff', data=simulated_destination, color='orange', fill=True, label='Simulated')
# sns.histplot(x='log_income_diff', data=actual_destination, color='teal', fill=True, label='Actual')

# plt.legend()
# plt.show()

In [ ]:
actual_destination_filtered = actual_destination[actual_destination['Familiarity_log'] >= 2.5]
simulated_destination_filtered = simulated_destination[simulated_destination['Familiarity_log'] >= 2.5]

plt.figure(figsize=(4, 4))

sns.histplot(
    x='Familiarity_log', 
    data=actual_destination_filtered, 
    label='Actual Destinations', 
    color='darkorange', 
    fill=True, 
    edgecolor='black', 
    linewidth=0.0, 
    bins=25
)
sns.histplot(
    x='Familiarity_log', 
    data=simulated_destination_filtered, 
    label='Random Destination', 
    color='lightgrey', 
    fill=True, 
    edgecolor='black', 
    linewidth=0.0, 
    bins=25
)

mean_actual = actual_destination_filtered['Familiarity_log'].mean()
mean_random = simulated_destination_filtered['Familiarity_log'].mean()

plt.axvline(mean_actual, color='darkorange', linestyle='-', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='darkgrey', linestyle='-', label='Mean Random Dest.', linewidth=1)

plt.xlim(10, 18)
plt.xlabel('Familiarity')
plt.gca().axes.yaxis.set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.gca().spines['bottom'].set_linewidth(0.5)
plt.savefig('hist_familiarity_5v.png', dpi=300)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import ttest_ind
actual_values = actual_destination_filtered['Familiarity_log']
random_values = simulated_destination_filtered['Familiarity_log']
t_stat, p_value = ttest_ind(actual_values, random_values, equal_var=False)  # Use equal_var=False for Welch's t-test

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
plt.figure(figsize=(4, 4))

sns.histplot(
    x='homophily', 
    data=actual_destination, 
    label='Actual Destinations', 
    color='steelblue', 
    fill=True, 
    edgecolor='black', 
    linewidth=0.0, 
    bins=25
)
sns.histplot(
    x='Similarity', 
    data=simulated_destination, 
    label='Random Destination', 
    color='lightgrey', 
    fill=True, 
    edgecolor='black', 
    linewidth=0.0, 
    bins=25
)

mean_actual = actual_destination['homophily'].mean()
mean_random = simulated_destination['Similarity'].mean()

plt.axvline(mean_actual, color='steelblue', linestyle='-', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='darkgrey', linestyle='-', label='Mean Random Dest.', linewidth=1)
plt.gca().axes.yaxis.set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.gca().spines['bottom'].set_linewidth(0.5)
plt.xlabel('Similarity')
plt.tight_layout()
plt.savefig('hist_similarity_5v.png', dpi=300)
plt.show()

In [ ]:
actual_values = actual_destination_filtered['homophily']
random_values = simulated_destination_filtered['Similarity']
t_stat, p_value = ttest_ind(actual_values, random_values, equal_var=False)  # Use equal_var=False for Welch's t-test

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
combined_act_sim = pd.concat([
    actual_destination.assign(destination_type='Actual Destinations'),
    simulated_destination.assign(destination_type='Random Destinations')
])
combined_act_sim = combined_act_sim.sort_values(by='Unnamed: 0.1').reset_index(drop=True)

combined_act_sim.head()

In [ ]:
actual_data_sim = combined_act_sim[combined_act_sim['destination_type'] == 'Actual Destinations']['homophily']
random_data_sim = combined_act_sim[combined_act_sim['destination_type'] == 'Random Destinations']['Similarity']

In [ ]:
actual_data_sim_mean = np.mean(actual_data_sim)
random_data_sim_mean = np.mean(random_data_sim)
actual_data_sim_median = np.std(actual_data_sim)/np.sqrt(len(actual_data_sim))
random_data_sim_median = np.std(random_data_sim)/np.sqrt(len(random_data_sim))


print(f"Actual Destinations: {actual_data_sim_mean, actual_data_sim_median}")
print(f"Random Destinations: {random_data_sim_mean, random_data_sim_median}")

In [ ]:
plt.figure(figsize=(5, 3))

# Create the boxplot
box = plt.boxplot(
    [actual_data_sim, random_data_sim],
    patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'],
    showfliers=False  # Hide outliers
)

# Set colors for the boxes
colors = ['steelblue', 'steelblue']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')

# Set the color of the median lines inside the boxes to dark blue
for median in box['medians']:
    median.set_color('darkblue')  # Change the median line color to dark blue
    median.set_linewidth(2)       # Make the median line thicker

# Add vertical dashed lines for the medians
for median in box['medians']:
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color='darkblue', linestyle='--', linewidth=1, zorder=5)  # Vertical line at the median

# Add light grey dashed grid lines
plt.grid(axis='x', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
plt.grid(axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)

# Set x-axis limits with padding to avoid clipping
plt.xlim(-1.1, 1.1)  # Add a small buffer to ensure no fading of error bars

# Remove plot spines
ax = plt.gca()  # Get current axis
for spine in ax.spines.values():
    spine.set_visible(False)

# Tight layout and save the plot
plt.tight_layout()
plt.savefig('boxplot_similarity.png', dpi=300)
plt.show()

In [ ]:
# plt.figure(figsize=(5, 3))
# box = plt.boxplot([actual_data_sim, random_data_sim], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

# colors = ['steelblue', 'steelblue']
# for patch, color in zip(box['boxes'], colors):
#     patch.set_facecolor(color)
#     patch.set_edgecolor('none')  
    
# for median, color in zip(box['medians'], colors):
#     median_value = median.get_xdata()[0]  # Get the median value position
#     plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

# for flier in box['fliers']:
#     flier.set(marker='')

# plt.xlim(-1, 1)

# plt.tight_layout()

# # Force the plot to render before saving
# plt.draw()

# # Save the plot in 300 DPI

# plt.savefig('boxplot_Similarity.png', dpi=300)

# plt.show()

In [ ]:
actual_data_fam = combined_act_sim[combined_act_sim['destination_type'] == 'Actual Destinations']['Familiarity_log']
random_data_fam = combined_act_sim[combined_act_sim['destination_type'] == 'Random Destinations']['Familiarity_log']

In [ ]:
actual_data_sim_mean = np.mean(actual_data_fam)
random_data_sim_mean = np.mean(random_data_fam)
actual_data_sim_median = np.median(actual_data_fam)
random_data_sim_median = np.median(random_data_fam)

print(f"Actual Destinations: {actual_data_sim_mean, actual_data_sim_median}")
print(f"Random Destinations: {random_data_sim_mean, random_data_sim_median}")

In [ ]:
import matplotlib.pyplot as plt

# Remove NaN values from both datasets
actual_data_fam_clean = actual_data_fam.dropna()
random_data_fam_clean = random_data_fam.dropna()

# Create a box plot using Matplotlib with cleaned data
plt.figure(figsize=(5, 3))

# Box plot without scatter points (outliers)
box = plt.boxplot(
    [actual_data_fam_clean, random_data_fam_clean], 
    patch_artist=True, vert=False, widths=0.6, 
    labels=['Actual Destinations', 'Random Destinations'], 
    showfliers=False  # Hide outliers
)

# Customizing the appearance of the box plot
colors = ['sandybrown', 'sandybrown']  # Colors for boxes
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')  # Remove the boundary

# Customizing the medians (set color to chocolate)
for median in box['medians']:
    median.set_color('chocolate')  # Set median color to chocolate
    median.set_linewidth(2)        # Make the median line thicker

# Add vertical dashed lines at median positions
for median in box['medians']:
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color='chocolate', linestyle='--', linewidth=1, zorder=5)  # Vertical line at median

# Set x-axis limits to ensure error bars are not cut
plt.xlim(7.5, 20)  # Adjust based on the data range

# Add light grey dashed grid lines
plt.grid(axis='x', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
plt.grid(axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)

# Remove bounding box (spines)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

# Add labels and title
#plt.xlabel('Familiarity (Raw Values)', fontsize=12)
#plt.title('Box Plot of Familiarity for Actual and Random Destinations (Cleaned Data)', fontsize=14)

# Adjust layout and save the plot
plt.tight_layout()
plt.savefig('boxplot_familiarity_no_outliers.png', dpi=300)
plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# # Remove NaN values from both datasets
# actual_data_fam_clean = actual_data_fam.dropna()
# random_data_fam_clean = random_data_fam.dropna()

# # Create a box plot using Matplotlib with cleaned data
# plt.figure(figsize=(5, 3))

# # Box plot with outliers shown and customized width
# box = plt.boxplot([actual_data_fam_clean, random_data_fam_clean], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

# # Customizing the appearance of the box plot
# colors = ['sandybrown', 'sandybrown']  # Different colors for actual and random

# # Change box colors and remove the edge color (no boundary)
# for patch, color in zip(box['boxes'], colors):
#     patch.set_facecolor(color)
#     patch.set_edgecolor('none')  # Remove the boundary

# # Show the outliers clearly
# for flier in box['fliers']:
#     flier.set(marker='')  # Set outlier appearance

# # Extend the median line vertically through the entire plot
# for median, color in zip(box['medians'], colors):
#     median_value = median.get_xdata()[0]  # Get the median value position
#     plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

# # Set appropriate x-axis limits to fit the range of your data
# plt.xlim(7.5, 20)  # Adjust based on the data range

# # Add labels and title
# plt.xlabel('Familiarity (Raw Values)')
# plt.title('Box Plot of Familiarity for Actual and Random Destinations (Cleaned Data)')

# plt.tight_layout()

# # Force the plot to render before saving
# plt.draw()

# # Save the plot in 300 DPI
# plt.savefig('boxplot_familiarity.png', dpi=300)

# plt.show()

<!-- ## import seaborn as sns

# Create a KDE plot for both datasets
plt.figure(figsize=(7, 3))

# KDE plot for actual destinations
sns.kdeplot(actual_data_fam, label='Actual Destinations', color='orange', fill=True)

# KDE plot for random destinations
sns.kdeplot(random_data_fam, label='Random Destinations', color='teal', fill=True)

# Add labels and title
plt.xlabel('Connectedness (Raw Values)')
plt.ylabel('Density')
plt.title('KDE Plot of Connectedness for Actual and Random Destinations')

# Add a legend
plt.legend()

# Show the plot
plt.tight_layout()
plt.show() -->

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(actual_data_fam, bins=10, alpha=0.7, label='Actual Destinations', color='orange')
plt.hist(random_data_fam, bins=10, alpha=0.7, label='Random Destinations', color='teal')
plt.xlabel('Familiarity (Raw Values)')
plt.ylabel('Frequency')
plt.title('Histogram of Familiarity for Actual and Random Destinations')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(actual_data_sim, bins=10, alpha=0.7, label='Actual Destinations', color='orange')
plt.hist(random_data_sim, bins=10, alpha=0.7, label='Random Destinations', color='teal')
plt.xlabel('Similarity (Raw Values)')
plt.ylabel('Frequency')
plt.title('Histogram of Familiarity for Actual and Random Destinations')
plt.legend()
plt.tight_layout()
plt.show()

# Sub group Plots

In [ ]:
actual_destination = pd.read_csv("evacuees_newSCI_simulation.csv").reset_index(drop=True)
actual_destination = actual_destination.rename(columns = {'homophily' : 'Similarity'})
actual_destination = actual_destination.rename(columns = {'scaled_sci' : 'Familiarity'})
actual_destination['Familiarity_log'] = actual_destination['Familiarity'].apply(np.log1p)
actual_destination.columns

In [ ]:
# evacuees_newSCI_cleaned['log_dist_epicentre'] = evacuees_newSCI_cleaned['dist_epicentre'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_distance_OD'] = evacuees_newSCI_cleaned['distance_OD'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_D1_population_density'] = evacuees_newSCI_cleaned['D1_population_density'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_population_density'] = evacuees_newSCI_cleaned['Or_population_density'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_unemployment_rate'] = evacuees_newSCI_cleaned['Or_unemployment_rate'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_total_housing_units'] = evacuees_newSCI_cleaned['Or_total_housing_units'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_radius_of_gyration'] = evacuees_newSCI_cleaned['radius_of_gyration'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_white_population'] = evacuees_newSCI_cleaned['Or_white_population'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_black_population'] = evacuees_newSCI_cleaned['Or_black_population'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_asian_population'] = evacuees_newSCI_cleaned['Or_asian_population'].apply(lambda x: np.log10(1 + x))
# evacuees_newSCI_cleaned['log_Or_all_others'] = evacuees_newSCI_cleaned['Or_all_others'].apply(lambda x: np.log10(1 + x))

In [ ]:
simulated_destination = pd.read_csv("simulated_location_revised.csv.gz")
simulated_destination['Familiarity_log'] = simulated_destination['scaled_sci'].apply(np.log1p)
simulated_destination.head()

In [ ]:
census_org = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")
census = census_org[(census_org['median_household_income'])>0]

In [ ]:
census['white_population'] = (census['white_population']  /census['total_population'] )*100
census['black_population'] = (census['black_population']  /census['total_population'] )*100
census['asian_population'] = (census['asian_population']  /census['total_population'] )*100
census['all_others'] = ((census['total_population']-(census['white_population']+census['black_population']+ census['asian_population']))
                                                    /census['total_population'])*100
census['education_atleast_one_degree'] = (census['total_population_25_over']  /census['total_population'] )*100

In [ ]:
columns_keep = ['total_housing_units','white_population', 'black_population', 'asian_population', 'education_atleast_one_degree', 
                'unemployment_rate', 'geoid', 'all_others', 'median_household_income' ]
census = census[columns_keep]
census.head()

In [ ]:
census_D1 = census.rename(
    columns=lambda x: 'Or_' + x if x != 'geoid' else x
)
census_D1.head()

In [ ]:
census_D1['geoid'] = census_D1['geoid'].astype(str)
simulated_destination['pre_crisis_home_cbg'] = simulated_destination['pre_crisis_home_cbg'].astype(str)

In [ ]:
simulated_destination = simulated_destination.merge(census_D1, left_on = "pre_crisis_home_cbg", right_on = "geoid", how = "inner")

In [ ]:
simulated_destination.columns

In [ ]:
white_population_median_actual = actual_destination['Or_white_population'].median()
black_population_median_actual = actual_destination['Or_black_population'].median()
median_income_median_actual = actual_destination['Or_median_household_income'].median()

white_population_median_simulated = simulated_destination['Or_white_population'].median()
black_population_median_simulated = simulated_destination['Or_black_population'].median()
median_income_median_simulated = simulated_destination['Or_median_household_income'].median()

In [ ]:
actual_destination['high_white_population'] = actual_destination['Or_white_population'] >= white_population_median_actual
actual_destination['high_black_population'] = actual_destination['Or_black_population'] >= black_population_median_actual
actual_destination['high_median_income'] = actual_destination['Or_median_household_income'] >= median_income_median_actual

simulated_destination['high_white_population'] = simulated_destination['Or_white_population'] >= white_population_median_simulated
simulated_destination['high_black_population'] = simulated_destination['Or_black_population'] >= black_population_median_simulated
simulated_destination['high_median_income'] = simulated_destination['Or_median_household_income'] >= median_income_median_simulated

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def combined_box_plots(actual_data, simulated_data):
    # Calculate overall median values for actual and simulated data
    overall_median_actual_familiarity = actual_data['Familiarity_log'].median()
    overall_median_simulated_familiarity = simulated_data['Familiarity_log'].median()
    overall_median_actual_similarity = actual_data['Similarity'].median()
    overall_median_simulated_similarity = simulated_data['Similarity'].median()

    # Define group filters
    group_filters = {
        "High White Population": (actual_data['high_white_population'], simulated_data['high_white_population']),
        #"Low White Population": (~actual_data['high_white_population'], ~simulated_data['high_white_population']),
        "High Black Population": (actual_data['high_black_population'], simulated_data['high_black_population']),
        #"Low Black Population": (~actual_data['high_black_population'], ~simulated_data['high_black_population']),
        "High Median Income": (actual_data['high_median_income'], simulated_data['high_median_income']),
        "Low Median Income": (~actual_data['high_median_income'], ~simulated_data['high_median_income'])
    }

    # Prepare data for Familiarity box plot
    familiarity_data = []
    familiarity_labels = []
    for group_name, (filter_actual, filter_simulated) in group_filters.items():
        # Add filtered data for actual and simulated groups with labels
        familiarity_data.append(actual_data[filter_actual]['Familiarity_log'].dropna())
        familiarity_labels.append(f"{group_name} (Actual)")
        familiarity_data.append(simulated_data[filter_simulated]['Familiarity_log'].dropna())
        familiarity_labels.append(f"{group_name} (Random)")

    # Plot all Familiarity box plots in one figure
    plt.figure(figsize=(10, 6))
    box = plt.boxplot(familiarity_data, patch_artist=True, vert=True, widths=0.6, labels=familiarity_labels, showfliers=False)
    
    # Set box colors to chocolate brown and median colors to orange (actual) and teal (random)
    for i, patch in enumerate(box['boxes']):
        if i % 2 == 0:  # Actual
            patch.set_facecolor('orange')
            patch.set_edgecolor('none')
            plt.setp(box['medians'][i], color='red', linestyle='-', linewidth=1.5)
        else:  # Random
            patch.set_facecolor('orange')
            patch.set_edgecolor('none')
            plt.setp(box['medians'][i], color='blue', linestyle='-', linewidth=1.5)

    # Add horizontal lines for overall medians
    plt.axhline(overall_median_actual_familiarity, color='red', linestyle='--', linewidth=1, label='Overall Median (Actual)')
    plt.axhline(overall_median_simulated_familiarity, color='blue', linestyle='--', linewidth=1, label='Overall Median (Random)')

    # Customize plot
    plt.ylabel('Familiarity (Log-Scaled Values)')
    plt.title('Combined Box Plot of Familiarity for Different Demographic Groups')
    plt.xticks(rotation=90, ha="right")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    # Prepare data for Similarity box plot
    similarity_data = []
    similarity_labels = []
    for group_name, (filter_actual, filter_simulated) in group_filters.items():
        # Add filtered data for actual and simulated groups with labels
        similarity_data.append(actual_data[filter_actual]['Similarity'].dropna())
        similarity_labels.append(f"{group_name} (Actual)")
        similarity_data.append(simulated_data[filter_simulated]['Similarity'].dropna())
        similarity_labels.append(f"{group_name} (Random)")

    # Plot all Similarity box plots in one figure
    plt.figure(figsize=(10, 6))
    box = plt.boxplot(similarity_data, patch_artist=True, vert=True, widths=0.6, labels=similarity_labels, showfliers=False)
    
    # Set box colors to steel blue and median colors to orange (actual) and teal (random)
    for i, patch in enumerate(box['boxes']):
        if i % 2 == 0:  # Actual
            patch.set_facecolor('steelblue')
            patch.set_edgecolor('none')
            plt.setp(box['medians'][i], color='red', linestyle='-', linewidth=1.5)
        else:  # Random
            patch.set_facecolor('steelblue')
            patch.set_edgecolor('none')
            plt.setp(box['medians'][i], color='blue', linestyle='-', linewidth=1.5)

    # Add horizontal lines for overall medians
    plt.axhline(overall_median_actual_similarity, color='red', linestyle='--', linewidth=1, label='Overall Median (Actual)')
    plt.axhline(overall_median_simulated_similarity, color='blue', linestyle='--', linewidth=1, label='Overall Median (Random)')

    # Customize plot
    plt.ylabel('Similarity (Raw Values)')
    plt.title('Combined Box Plot of Similarity for Different Demographic Groups')
    plt.xticks(rotation=90, ha="right")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
combined_box_plots(actual_destination, simulated_destination)